# 📚 Pipeline de Preparação de Dados e Engenharia de Atributos

**Projeto:** Análise e Predição de Alarmes (`Is_Dont_Go`)
**Fase:** Limpeza, Engenharia de Variáveis (Feature Engineering) e Formatação Estrutural

> 💡 **Objetivo Principal:** Aqui transformamos dados brutos numa matriz pronta para o consumo de algoritmos de Machine Learning. Executamos conversões matemáticas, tratamos nulos de forma refinada, codificamos categorias e criamos alvos temporais importantes (alvos preditivos para janelas de tempo futuro).

## 🔍 1. Configuração do Ambiente e Bibliotecas

Importação de pacotes necessários para manipulação algébrica (`numpy`, `pandas`), visualização (`matplotlib`, `seaborn`) e ferramentas do `scikit-learn` para escalonamento e codificação.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import scipy.stats as ss
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

In [3]:
!tree ../..


../..
├── Programa Desenvolver - Análise Avançada de Dados-20260502T151748Z-3-001
│   └── Programa Desenvolver - Análise Avançada de Dados
│       ├── Base de Dados
│       │   ├── Alarmes - Regra de Negocio.ods
│       │   ├── Alarmes - Regra de Negocio.xlsx
│       │   ├── datasets
│       │   │   ├── apontamentos
│       │   │   │   ├── desenvolver_apontamentos.parquet
│       │   │   │   └── desenvolver_apontamentos.xlsx
│       │   │   ├── README.md
│       │   │   └── telemetria
│       │   │       ├── desenvolver_dontgo.xlsx
│       │   │       ├── telemetry_abr.parquet
│       │   │       ├── telemetry_feb.parquet
│       │   │       ├── telemetry_jan.parquet
│       │   │       ├── telemetry_jun.parquet
│       │   │       ├── telemetry_mar.parquet
│       │   │       └── telemetry_may.parquet
│       │   ├── datasets.7z
│       │   └── Dicionario_Dados.xlsx
│       ├── Desenvolver_Template.docx
│       └── Estudo Guiado - Análise Avançada de Dados.pdf
├── Programa Desenvolver

## 🔍 2. Carregamento dos Dados Limpos

Resgatamos a versão do CSV resultante do fim da Fase 1, já isenta de colunas constantes, vazias e inúteis para o modelo.

In [22]:
path = os.getcwd()

# Defina o mês que deseja trabalhar (ex: 'jan', 'feb', 'marco')
months = ["jan", "feb", "marco", "apr", "may", "jun"]
mes = 0  # 5 -> 'jun'
mes_escolhido = months[mes] 

# AJUSTE: Subindo um nível para acessar a pasta 'data' a partir de 'notebooks'
arquivo_mes = os.path.join("..", "data", "telemetry", "processed", f"{mes_escolhido}.csv")

df = pd.read_csv(arquivo_mes, low_memory=False)
df.head()

,Id_Eventos_Telemetria,Data_Evento,Inicio_Turno,Fim_Turno,Dia,Localidade,TAG,Tag_Frota,Tipo,Nome_Operador_Anon,Matricula_Operador_Hash,Id_Alarme,Alarme,Id_Criticidade,Criticidade,Valor,Classe,Is_Dont_Go
0,9372078,2025-01-01 00:00:14.553000,2024-12-31 18:00:00.000,2025-01-01 06:00:00.000,1,Itabira,CA5926,793-D 5S,Caminhao,OP_261,H_7d2a33163e23,335609934,Rx Channel A Not Receiving Messages,3,Informacional,0,NaN,0
1,9372079,2025-01-01 00:00:14.553000,2024-12-31 18:00:00.000,2025-01-01 06:00:00.000,1,Itabira,CA5926,793-D 5S,Caminhao,OP_261,H_7d2a33163e23,335609935,Rx Channel B Not Receiving Messages,3,Informacional,0,NaN,0
2,9372097,2025-01-01 00:02:20.280000,2024-12-31 18:00:00.000,2025-01-01 06:00:00.000,1,Itabira,CA5926,793-D 5S,Caminhao,OP_261,H_7d2a33163e23,84626976,Dipper,3,Informacional,"43,7999992370605",NaN,0
3,9372111,2025-01-01 00:02:59.077000,2024-12-31 18:00:00.000,2025-01-01 06:00:00.000,1,Itabira,CA5926,793-D 5S,Caminhao,OP_261,H_7d2a33163e23,84626976,Dipper,3,Informacional,"74,7000045776367",NaN,0
4,9372120,2025-01-01 00:03:37.963000,2024-12-31 18:00:00.000,2025-01-01 06:00:00.000,1,Itabira,CA5926,793-D 5S,Caminhao,OP_261,H_7d2a33163e23,84626976,Dipper,3,Informacional,"103,800003051758",NaN,0


## 🔍 3. Transformação e Criação de Features

A essência do **Feature Engineering**. 

### 🛠️ O que será feito:
1. **Conversões Regionais:** Ajuste de vírgulas para pontos em variáveis numéricas, típico de bases latino-americanas.
2. **Tratamento de Janela de Tempo (ffill/bfill):** Os valores de telemetria por TAG de equipamento serão projetados para preencher lacunas utilizando informações passadas (Forward Fill) ou futuras próximas (Backward Fill).
3. **Tempo de Turno:** Cálculo de quantas horas a máquina operou, subtraindo datas e instanciando o período do dia (Madrugada, Manhã, Tarde).
4. **Codificação:** Transformação de colunas descritivas em indexadores numéricos usando o `LabelEncoder`.

In [23]:
# 1. Limpeza rigorosa da coluna Valor (padrão brasileiro) e NaNs por TAG
df['Valor'] = df['Valor'].astype(str).str.replace(',', '.')
df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')
df['Valor'] = df.groupby('TAG')['Valor'].ffill().bfill().fillna(0)

# 2. Conversão de Datas e Ordenação Temporal Absoluta por Equipamento
df['Data_Evento'] = pd.to_datetime(df['Data_Evento'])
df['Inicio_Turno'] = pd.to_datetime(df['Inicio_Turno'])
df = df.sort_values(by=['TAG', 'Data_Evento']).reset_index(drop=True)

# 3. Cálculo do Tempo de Turno Trabalhado
df['Tempo_Trabalhando_Horas'] = (df['Data_Evento'] - df['Inicio_Turno']).dt.total_seconds() / 3600

# 4. Cálculo do Período do Dia
horas = df['Data_Evento'].dt.hour
limites = [0, 4, 8, 12, 16, 20, 24]
rotulos = ['Noite 2', 'Manhã 1', 'Manhã 2', 'Tarde 1', 'Tarde 2', 'Noite 1']
df['Periodo_Dia'] = pd.cut(horas, bins=limites, labels=rotulos, right=False)
df['Periodo_Dia_Int'] = LabelEncoder().fit_transform(df['Periodo_Dia'].astype(str))

# 5. Transformação de variáveis textuais cruciais em IDs numéricos
le = LabelEncoder()
colunas_texto = ['TAG', 'Tag_Frota', 'Tipo']
for col in colunas_texto:
    df[col] = le.fit_transform(df[col].astype(str))

# 6. Mapeamento e Indexação de Operadores
operadores_unicos = df['Matricula_Operador_Hash'].unique()
dict_operadores = {hash_op: i for i, hash_op in enumerate(operadores_unicos)}
df['Matricula_Operador_Hash'] = df['Matricula_Operador_Hash'].map(dict_operadores)

## 🔍 4. Análise de Correlação (Cramér's V)

Como estamos lidando com variáveis em sua maioria categóricas para prever um alvo binário (`Is_Dont_Go`), a correlação de Pearson nem sempre é adequada.

> 💡 **Por que o V de Cramér?** Ele é uma medida estatística para variáveis nominais. Ajuda a entender a força da associação entre a criticidade do alarme, o tipo de equipamento e o nosso alvo, revelando quais colunas são cruciais para o modelo focar.

In [24]:
cols_numericas = ['Valor', 'Id_Criticidade']
print("--- Correlação com Is_Dont_Go (Variáveis Numéricas) ---")
for col in cols_numericas:
    corr, p_value = ss.pointbiserialr(df[col], df['Is_Dont_Go'])
    print(f"{col} -> Correlação: {corr:.4f} | P-value: {p_value:.4e}")

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = ss.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

cols_categoricas = ['TAG', 'Tag_Frota', 'Tipo', 'Alarme', 'Criticidade']
print("\n--- Força de Associação com Is_Dont_Go (Cramér's V) ---")
for col in cols_categoricas:
    if col in df.columns:
        valid_idx = df[col].notna() & df['Is_Dont_Go'].notna()
        v = cramers_v(df.loc[valid_idx, col], df.loc[valid_idx, 'Is_Dont_Go'])
        print(f"{col}: {v:.4f}")
print(len(df))

--- Correlação com Is_Dont_Go (Variáveis Numéricas) ---
Valor -> Correlação: -0.0018 | P-value: 2.0122e-05
Id_Criticidade -> Correlação: -0.2260 | P-value: 0.0000e+00

--- Força de Associação com Is_Dont_Go (Cramér's V) ---
TAG: 0.0929
Tag_Frota: 0.0747
Tipo: 0.0685
Alarme: 0.5449
Criticidade: 0.2524
5400002


## 🔍 5. Treino do Classificador Baseline (Regra Linear)

Para sabermos se o modelo avançado que criaremos depois (como Redes Neurais LSTM) é bom, precisamos ter uma referência (Baseline). Treinamos uma **Regressão Logística Básica** para obter as métricas iniciais sem otimizações pesadas.

In [25]:
print(len(df))
# Limpeza Funcional: Exclusão estrita das features não escalares ou de alvo direto que vazam informações para o modelo.
# colunas_remover = ['Is_Dont_Go', 'Data_Evento', 'Inicio_Turno', 'Periodo_Dia', 'Alarme', 'Criticidade', 'Fim_Turno']
# X_lr = df.drop(columns=[c for c in colunas_remover if c in df.columns])
# y_lr = df['Is_Dont_Go']

# modelo_lr = LogisticRegression(random_state=42, max_iter=1000)
# modelo_lr.fit(X_lr, y_lr)
# # 
# print("Classificador Baseline de Regressão Logística treinado com sucesso.")
# print(f"Formato da matriz de features X: {X_lr.shape}")

5400002


## 🔍 6. Reamostragem Temporal (Resampling) de Séries Temporais

Os alarmes não acontecem de forma cadenciada. Eles chegam espalhados, muitas vezes na mesma fração de segundo. Modelos de Machine Learning preferem estruturas regulares.

### 🛠️ Estratégia de Agrupamento:
Agruparemos o log por janelas cravadas de **10 em 10 minutos** (`resample('10min')`) por equipamento (`TAG`). Em caso de silêncio do sensor, propagaremos os estados conhecidos usando os métodos de preenchimento (`ffill`).

In [26]:
agg_dict = {
    'Tag_Frota': 'last', 'Tipo': 'last', 'Matricula_Operador_Hash': 'last',
    'Id_Alarme': 'last', 'Id_Criticidade': 'max', 'Valor': 'mean',                  
    'Tempo_Trabalhando_Horas': 'max', 'Periodo_Dia_Int': 'last', 'Is_Dont_Go': 'max'               
}

# Granularidade Temporal: Agrupamento em saltos rigorosos e interpolados de 10 minutos preservando estritamente a janela por TAG do Caminhão/Escavadeira.
df_resampled = (
    df.set_index('Data_Evento')
       .groupby('TAG')
       .resample('10min')
       .agg(agg_dict)
       .reset_index()
)


# Propagação para preenchimento de lacunas vazias entre passos de tempo
cols_to_fill = list(agg_dict.keys())
df_resampled[cols_to_fill] = df_resampled.groupby('TAG')[cols_to_fill].ffill().bfill()

# Restauração de tipos inteiros pós-ffill
int_cols = ['Tag_Frota', 'Tipo', 'Matricula_Operador_Hash', 'Id_Alarme', 'Id_Criticidade', 'Periodo_Dia_Int', 'Is_Dont_Go']
df_resampled[int_cols] = df_resampled[int_cols].astype(int)

## 🔍 7. Geração do Target Preditivo Antecipado

A ideia de uma predição inteligente não é prever a falha no momento em que ela ocorre, mas sim **horas antes**.

> 💡 **Lógica do Rolling Backward:** Para cada janela de 10 minutos, o código checa as próximas 4 horas (24 janelas à frente). Se houver uma falha nesse período no futuro, marcamos a célula do presente como `Target_Predictive = 1`. Isso treinará o modelo para identificar o padrão pre-falha.

In [27]:
# Configuração Preditiva: 4 horas de horizonte multiplicadas em blocos contínuos de 10 minutos perfazem 24 janelas completas (Passos de tempo para alvo futuro).
passos_futuros = 24

# Janela Rolante Reversa (Backward Rolling): Verifica se ao menos um "Don't Go" ocorrerá nos próximos 24 saltos de tempo. Usa reversão de array para captar o futuro.
df_resampled['Target_Predictive'] = (
    df_resampled.groupby('TAG')['Is_Dont_Go']
    .transform(lambda x: x.iloc[::-1].rolling(window=passos_futuros, min_periods=1).max().iloc[::-1])
)
df_resampled['Target_Predictive'] = df_resampled['Target_Predictive'].fillna(0).astype(int)

print(f"Falhas reais registadas: {df_resampled['Is_Dont_Go'].sum()}")
print(f"Janelas de alerta geradas (Target_Predictive): {df_resampled['Target_Predictive'].sum()}")
print(f"O tamanho de linhas do dataset atual é de {len(df)}")

Falhas reais registadas: 2134
Janelas de alerta geradas (Target_Predictive): 15850
O tamanho de linhas do dataset atual é de 5400002


## 🔍 8. Padronização de Colunas Contínuas

Algoritmos como Redes Neurais e Regressão Logística sofrem variações catastróficas se as escalas dos dados estiverem diferentes. Utilizamos o `StandardScaler` (média 0, desvio 1) para padronizar variáveis contínuas, garantindo fluidez no cálculo numérico das predições.

In [28]:
scaler = StandardScaler()
colunas_para_escalonar = ['Valor', 'Tempo_Trabalhando_Horas']

df_resampled[colunas_para_escalonar] = scaler.fit_transform(df_resampled[colunas_para_escalonar])
print("Colunas contínuas padronizadas com sucesso no df_resampled.")

Colunas contínuas padronizadas com sucesso no df_resampled.


## 🔍 9. Extração de Sazonalidade Semanal

Modelagem Temporal exige variáveis de controle cronológico. Desmembramos as datas em componentes de Dias da Semana (Segunda a Domingo), representadas em formato Numérico Binário. Há comportamentos sistêmicos ou de manutenção associados a finais de semana ou início de turnos.

In [29]:
# Captura o dia da semana (0 = Segunda, 6 = Domingo)
df_resampled['Dia_Semana'] = df_resampled['Data_Evento'].dt.dayofweek

# Criação dinâmica e vetorizada das variáveis binárias de sazonalidade
for i in range(1, 7):
    df_resampled[f'Dia_{i}'] = (df_resampled['Dia_Semana'] == (i-1)).astype(int)

df_resampled.drop(columns=['Dia_Semana'], inplace=True)

## 🔍 10. Persistência dos Dados Preparados

Os dados agora encontram-se rigorosamente padronizados, interpolados, escalonados e com alvos pre-processados. Salvamos o consolidado no disco como base sólida e determinística para os próximos treinamentos (ex: Redes Neurais LSTM).

In [30]:
colunas_lstm = [
    'Data_Evento', 'TAG', 'Tag_Frota', 'Tipo', 'Matricula_Operador_Hash',
    'Id_Alarme', 'Id_Criticidade', 'Valor', 'Tempo_Trabalhando_Horas',
    'Periodo_Dia_Int', 'Dia_1', 'Dia_2', 'Dia_3', 'Dia_4', 'Dia_5', 'Dia_6',
    'Is_Dont_Go', 'Target_Predictive'
]

# AJUSTE: Define a pasta destino correspondente ao mês dentro de 'notebooks/'
pasta_saida = os.path.join(".", mes_escolhido)
os.makedirs(pasta_saida, exist_ok=True) # Garante que a pasta (ex: ./jun) existe

caminho_salvar = os.path.join(pasta_saida, f"{mes_escolhido}_dados_lstm.csv")

# Exportação do Checkpoint Analítico
df_resampled[colunas_lstm].to_csv(caminho_salvar, index=False)
print(f"✅ Pipeline executado linearmente! Ficheiro {caminho_salvar} guardado com sucesso.")

✅ Pipeline executado linearmente! Ficheiro ./jan/jan_dados_lstm.csv guardado com sucesso.
